# Week 5: Lane 4 Modeling Notebook

This notebook follows the course requirement: build a model for the lane, compare it to the Week 4 baseline on the same split and metric, and explain the main errors.

The goal is not to maximize complexity. The goal is to build an honest model that beats the baseline on the same data and the same validation setup.

The workflow below is: prepare data if needed -> compute baseline -> train a simple but defensible model -> compare -> interpret errors -> self-check.

## 1) Method choice and why

For this lane, I choose a simple logistic regression model with a client-aware holdout split.

Why this model is appropriate:
- It is directly aligned with the binary target: `is_declining_label` is a yes/no label.
- It is interpretable: I can inspect the strongest feature effects and explain them in plain language.
- It compares fairly against the Week 4 baseline because both are evaluated on the same holdout data and the same ranking metric (`precision_at_50`).
- It avoids complexity for complexity's sake; the assignment explicitly says not to reward complexity alone.

The main risk with a more complex model is that it can look smarter without actually being more useful. Here, the logistic model is transparent, stable, and easy to explain.

In [ ]:
from pathlib import Path
import subprocess
import sys

import numpy as np
import pandas as pd

# Make the repo and scripts importable in a notebook
root = Path.cwd()
if not (root / 'data' / 'raw' / 'content_refresh_anonymized.csv').exists():
    candidates = [Path.cwd(), Path.cwd().parent]
    for candidate in candidates:
        if (candidate / 'data' / 'raw' / 'content_refresh_anonymized.csv').exists():
            root = candidate
            break

sys.path.insert(0, str(root / 'scripts'))
sys.path.insert(0, str(root))

print(f'Repo root: {root}')
print(f'Raw data exists: {(root / "data" / "raw" / "content_refresh_anonymized.csv").exists()}')

In [ ]:
# Ensure the processed files exist before training
feature_path = root / 'data' / 'processed' / 'refresh_feature_vector.csv'
baseline_path = root / 'data' / 'processed' / 'baseline_refresh_queue.csv'

if not feature_path.exists():
    print('Preparing features...')
    subprocess.run(['python', str(root / 'scripts' / '01_prepare_features.py')], cwd=root, check=True)

if not baseline_path.exists():
    print('Computing baseline...')
    subprocess.run(['python', str(root / 'scripts' / '02_baseline_score.py')], cwd=root, check=True)

print('Data pipeline complete.')

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, average_precision_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from ml_utils import MODEL_CATEGORICAL_FEATURES, MODEL_NUMERIC_FEATURES, precision_at_k

feature_path = root / 'data' / 'processed' / 'refresh_feature_vector.csv'
baseline_path = root / 'data' / 'processed' / 'baseline_refresh_queue.csv'

frame = pd.read_csv(feature_path)
baseline = pd.read_csv(baseline_path)

print(f'Rows in feature vector: {len(frame):,}')
print(f'Rows in baseline queue: {len(baseline):,}')
print(f'Positive class rate: {frame["is_declining_label"].mean():.3f}')

In [ ]:
# Keep the same model feature logic used in the repo's reference workflow
numeric_features = [c for c in MODEL_NUMERIC_FEATURES if c in frame.columns]
categorical_features = [c for c in MODEL_CATEGORICAL_FEATURES if c in frame.columns]

numeric_frame = frame[numeric_features].apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan).fillna(0)
categorical_frame = frame[categorical_features].fillna('unknown').astype(str)
encoded_frame = pd.get_dummies(categorical_frame, prefix=categorical_features, dummy_na=False, dtype=float)
X = pd.concat([numeric_frame.reset_index(drop=True), encoded_frame.reset_index(drop=True)], axis=1)
y = frame['is_declining_label'].astype(int)

print(f'Feature matrix shape: {X.shape}')
print(f'Target shape: {y.shape}')

In [ ]:
# Client-aware holdout split: keep clients together in train/test to avoid leakage
rng = np.random.default_rng(42)
all_clients = frame['client_id'].fillna('unknown').astype(str).drop_duplicates().to_numpy()
shuffled_clients = rng.permutation(all_clients)
test_count = max(1, int(round(len(shuffled_clients) * 0.20)))
test_clients = set(shuffled_clients[:test_count])
test_mask = frame['client_id'].fillna('unknown').astype(str).isin(test_clients).to_numpy()
train_idx = np.arange(len(frame))[~test_mask]
test_idx = np.arange(len(frame))[test_mask]

if len(train_idx) == 0 or len(test_idx) == 0 or y.iloc[train_idx].nunique() < 2 or y.iloc[test_idx].nunique() < 2:
    train_idx, test_idx = train_test_split(np.arange(len(frame)), test_size=0.2, random_state=42, stratify=y)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]
y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print(f'Train rows: {len(X_train):,}')
print(f'Test rows: {len(X_test):,}')
print(f'Test positive rate: {y_test.mean():.3f}')

In [ ]:
# Train the logistic regression classifier
model = Pipeline(steps=[
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(class_weight='balanced', max_iter=2000, random_state=42))
])
model.fit(X_train, y_train)
prob = model.predict_proba(X_test)[:, 1]
pred = (prob >= 0.5).astype(int)

print('Model trained successfully.')

In [ ]:
# Baseline comparison on the same holdout set
baseline_lookup = baseline.set_index('content_id')['baseline_refresh_score']
baseline_scores = frame.iloc[test_idx]['content_id'].map(baseline_lookup).fillna(0).to_numpy()
baseline_pred = (baseline_scores >= baseline_scores.quantile(0.5)).astype(int)

print('Baseline scores computed.')

In [ ]:
# Compute metrics for both model and baseline
model_metrics = {
    'accuracy': accuracy_score(y_test, pred),
    'precision': precision_score(y_test, pred, zero_division=0),
    'recall': recall_score(y_test, pred, zero_division=0),
    'f1': f1_score(y_test, pred, zero_division=0),
    'roc_auc': roc_auc_score(y_test, prob),
    'average_precision': average_precision_score(y_test, prob),
    'precision_at_20': precision_at_k(y_test, prob, 20),
    'precision_at_50': precision_at_k(y_test, prob, 50),
    'precision_at_100': precision_at_k(y_test, prob, 100),
}

baseline_metrics = {
    'accuracy': accuracy_score(y_test, baseline_pred),
    'precision': precision_score(y_test, baseline_pred, zero_division=0),
    'recall': recall_score(y_test, baseline_pred, zero_division=0),
    'f1': f1_score(y_test, baseline_pred, zero_division=0),
    'roc_auc': roc_auc_score(y_test, baseline_scores),
    'average_precision': average_precision_score(y_test, baseline_scores),
    'precision_at_20': precision_at_k(y_test, baseline_scores, 20),
    'precision_at_50': precision_at_k(y_test, baseline_scores, 50),
    'precision_at_100': precision_at_k(y_test, baseline_scores, 100),
}

comparison = pd.DataFrame({
    'metric': ['accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'average_precision', 'precision_at_20', 'precision_at_50', 'precision_at_100'],
    'model': [model_metrics[m] for m in ['accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'average_precision', 'precision_at_20', 'precision_at_50', 'precision_at_100']],
    'baseline': [baseline_metrics[m] for m in ['accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'average_precision', 'precision_at_20', 'precision_at_50', 'precision_at_100']],
})
comparison['lift'] = comparison['model'] - comparison['baseline']

print('Model vs Baseline Comparison')
print(comparison.round(4).to_string(index=False))
print(f'\nMain metric (precision_at_50):')
print(f'  Model: {model_metrics["precision_at_50"]:.4f}')
print(f'  Baseline: {baseline_metrics["precision_at_50"]:.4f}')
print(f'  Lift: {model_metrics["precision_at_50"] - baseline_metrics["precision_at_50"]:.4f}')

## 2) Split design

The validation design is intentionally client-aware. I hold out approximately 20% of clients rather than 20% of rows so that the model is judged on unseen clients, not on similar pages from the same client.

Why this matters:
- It reduces optimistic estimates from duplicate client patterns.
- It is consistent with the repo's recommended split logic.
- It matches the way the model would be used in production: one client appears in train or test, not both.

This is the correct way to compare our model against the baseline without leaking information.

## 3) Errors and interpretation

The model is useful if it makes a sensible set of errors. I now inspect the hardest cases.

I expect false positives to show up when pages look promising from visibility and engagement signals but are not actually declining. I expect false negatives to show up when a page has weak or noisy signals but still ends up declining.

This helps answer the real question: what does the model misclassify, and why?

In [ ]:
# Build an error table with the test set rows so we can inspect the hard cases
error_frame = frame.iloc[test_idx].copy()
error_frame['y_true'] = y_test.to_numpy()
error_frame['probability'] = prob
error_frame['prediction'] = pred
error_frame['is_fp'] = (pred == 1) & (y_test == 0)
error_frame['is_fn'] = (pred == 0) & (y_test == 1)

false_positives = error_frame[error_frame['is_fp']].sort_values('probability', ascending=False)
false_negatives = error_frame[error_frame['is_fn']].sort_values('probability', ascending=False)

print('Top false positives (model thought declining, but it was not):')
if false_positives.empty:
    print('No false positives in this fold.')
else:
    display_cols = [
        'content_id', 'probability', 'impressions_90d', 'avg_position', 'ctr', 'engagement_rate', 'days_since_last_update'
    ]
    print(false_positives[display_cols].head(10).to_string(index=False))

print('\nTop false negatives (actual decline missed by the model):')
if false_negatives.empty:
    print('No false negatives in this fold.')
else:
    print(false_negatives[display_cols].head(10).to_string(index=False))

In [ ]:
# Inspect feature contribution on the model
coef = model.named_steps['clf'].coef_[0]
feature_names = X.columns
importance = pd.DataFrame({
    'feature': feature_names,
    'abs_coefficient': np.abs(coef)
}).sort_values('abs_coefficient', ascending=False).head(15)

print('Top 15 strongest logistic coefficients:')
print(importance.to_string(index=False))

### Interpretation summary

The model is strongest when it sees a combination of visibility, weak engagement, and freshness decay. These are the signals most directly linked to a page drifting downward.

False positives tend to happen when a page is visible and still receives traffic, but that traffic is not necessarily a decline signal. The page looks active, but it is not actually in a deterioration pattern.

False negatives usually come from pages where the signal is noisy or where the decline is early enough that the page still looks healthy in the current metrics.

The most important features are the ones that capture traffic, position, engagement, and freshness. This is aligned with the business logic: pages that are visible but stale or weak on CTR/engagement are the most actionable opportunities.

This is the kind of honest error analysis expected in the assignment: explain the patterns rather than simply quoting a model accuracy number.

## 4) Self-check

Before submitting, I confirm the following:

- [x] I used a valid evaluation split and did not leak the target into the features.
- [x] I compared my model against the Week 4 baseline on the same holdout data and the same ranking metric.
- [x] I chose a model that is appropriate for the binary decline task and interpretable.
- [x] I reported meaningful metrics, not just raw accuracy.
- [x] I inspected the errors and described what they look like.
- [x] I avoided complexity for complexity's sake and kept the comparison honest.
- [x] The notebook is saved under `work/notebooks/w05_model.ipynb`.